In [123]:
import openai
import langchain
import pinecone
from langchain_classic.chains.question_answering import load_qa_chain
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader,PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import Pinecone, PineconeVectorStore

In [171]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [172]:
## Read the document
def read_doc(directory):
    file_loader = PyPDFDirectoryLoader(directory)
    documents = file_loader.load()
    return documents

In [173]:
doc = read_doc('LLM Generic app/document')
len(doc)

58

In [174]:
## Divide the docs into chunks
def chunk_data(docs,chunk_size=700,chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap)
    doc = text_splitter.split_documents(docs)
    return doc

In [175]:
chunkDoc = chunk_data(doc)
chunkDoc[50]

Document(metadata={'producer': 'Adobe Acrobat Pro 10.1.16', 'creator': 'Adobe Acrobat Pro 10.1.16', 'creationdate': '2023-02-01T05:28:04+05:30', 'moddate': '2023-02-01T08:28:21+05:30', 'title': '', 'source': 'LLM Generic app/document/budget_speech.pdf', 'total_pages': 58, 'page': 19, 'page_label': '20'}, page_content='Stack and UPI. To enable more Fintech innovative services, the scope of \ndocuments available in DigiLocker for individuals will be expanded.  \nEntity DigiLocker  \n72. An Entity DigiLocker will be set up for use by MSMEs, large business \nand charitable trusts. This will be towards storing and sharing documents \nonline securely, whenever needed, with various authorities, regulators, \nbanks and other business entities.  \n5G Services \n73. One hundred labs for developing applications using  \n5G services will be set up in engineering institutions to realise a new range \nof opportunities, business models, and employment potential. The  labs will')

In [176]:
#Embedding technique
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4724.50it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [177]:
vectors = embeddings.embed_query("hello world")
vectors[1:10]

[0.03102320432662964,
 0.0067349388264119625,
 0.0261089950799942,
 -0.03936199098825455,
 -0.16030247509479523,
 0.06692396104335785,
 -0.006441457662731409,
 -0.04745052382349968,
 0.014758871868252754]

In [178]:
from pinecone import Pinecone

pc = Pinecone(
    api_key=os.getenv("PINECONE_API_KEY")
)

index = pc.Index("langchainvectordb")

In [179]:
index

In [180]:
vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings
)

In [181]:
vectorstore.add_documents(chunkDoc)

['a94ea090-6cbc-4484-9de4-3c8f4b377111',
 'ce255e4d-ffd0-4d24-933d-1d487dd5bc45',
 'e61c8c2a-506f-433a-bb49-f5c9708050c2',
 'b2a3c759-1f23-4438-b794-15cd390bc5f7',
 '29a02e69-06c3-4399-9b31-bce8e2c2ad60',
 'a26bacaf-07ba-4a47-aa4c-16db88fe3d94',
 '87af75b0-9cf8-429b-9e02-09fd9fe20252',
 'a84fa8db-fdf2-4dd9-b9ed-c3867a8ed941',
 'eeda832e-ece2-4ce2-a5fe-396946f483ce',
 '91394b18-0450-4fa9-9a0c-1e2f840b8050',
 'e608aa86-0c31-4bc1-930e-20a5a4a2b311',
 '2da63782-4b96-4999-8545-1928d2b28cab',
 '09b5a996-e52e-4fd1-9a52-3c4bbfad043c',
 '2e40c668-39ae-4625-b15e-f09515e74a05',
 'c02a9298-0439-4c93-84fa-0573795d4745',
 'd23c33e4-3b6d-47d3-a8c8-0193329aec04',
 'ff9df5ba-ba78-4fc4-9169-5d85c0963431',
 '0052c8af-bfee-4a2e-9d2b-92db3f59b110',
 'a15fd4b9-5013-45ec-8bd5-a09d287fd79e',
 '09defc52-b70e-4bc1-80a6-48c4b0330ae1',
 '00316f4e-19b2-458e-8334-b593ec4dbbb8',
 '810d80cf-5050-43ef-b259-3630a302e92e',
 'c58bba24-a039-4021-9467-0367b9d33552',
 'a5643a6c-97a4-47b0-86fd-14e9517d8709',
 '628089a1-b0c4-

In [182]:
#Cosine similarity retrieve results from vectorDB
def retrieve_query(query, k=2):
    matching_results = vectorstore.similarity_search(query,k=k)
    return matching_results

In [183]:
from langchain_groq import ChatGroq
from langchain_openai import OpenAI
from langchain_classic.chains.question_answering import load_qa_chain
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.5
)
chain = load_qa_chain(llm,chain_type='stuff')

In [184]:
def retrieve_answers(query):
    doc_search = retrieve_query(query)
    print(doc_search)
    response=chain.run(input_documents = doc_search,question=query)
    return response

In [185]:
our_query = "How much the agriculture target will be increased by how many crores?"
answer = retrieve_answers(our_query)

print(answer)

[Document(id='5dd4bf9e-c764-4201-9797-955698e70e0b', metadata={'creationdate': '2023-02-01T05:28:04+05:30', 'creator': 'Adobe Acrobat Pro 10.1.16', 'moddate': '2023-02-01T08:28:21+05:30', 'page': 10.0, 'page_label': '11', 'producer': 'Adobe Acrobat Pro 10.1.16', 'source': 'LLM Generic app/document/budget_speech.pdf', 'title': '', 'total_pages': 58.0}, page_content="7 \n \n \n \nfarmers in contributing to the health of fellow citizens by growing these \n‘Shree Anna’.  \n22. Now to make India a global hub for ' Shree Anna', the Indian Institute \nof Millet Research, Hyderabad will be supported as the Centre of Excellence \nfor sharing best practices, research and technologies at the international \nlevel.    \nAgriculture Credit  \n23. The agriculture credit target will be increased  \nto ` 20 lakh crore with focus on animal husbandry, dairy and fisheries.  \nFisheries \n24. We will launch a new sub-scheme of PM Matsya Sampada Yojana \nwith targeted investment of ` 6,000 crore to further